# BirdCLEF+ 2026 — Inference & Submission Notebook

**This notebook is submitted to Kaggle for evaluation.**

Configuration for this notebook:
- Internet: OFF
- Accelerator: None (CPU only)
- Loads model weights from an attached Kaggle Dataset

**Input Data:**
1. BirdCLEF 2026 competition data
2. Trained model dataset containing model_fold0.pt

In [1]:
import os, gc, time, warnings
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
import sys

warnings.filterwarnings('ignore')
START_TIME = time.time()
DEVICE = torch.device('cpu')
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

Device: cpu | PyTorch: 2.10.0+cpu


In [2]:
class CFG:
    # Competition data
    BASE_DIR       = Path('/kaggle/input/competitions/birdclef-2026')
    TEST_AUDIO_DIR = BASE_DIR / 'test_soundscapes'
    TRAIN_CSV      = BASE_DIR / 'train.csv'
    SAMPLE_SUB     = BASE_DIR / 'sample_submission.csv'

    # Pre-trained and published model dataset path - model_fold0.pt
    MODEL_PATHS    = [
        '/kaggle/input/datasets/saisamyukthan/birdclef2026-v1-efficientnet-b0-pretrained-outputs/model_fold0.pt',
    ]

    # Must match training CFG exactly
    SAMPLE_RATE    = 32000
    WINDOW_SIZE    = 5
    N_FFT          = 1024
    HOP_LENGTH     = 64
    N_MELS         = 136
    FMIN           = 20
    FMAX           = 16000
    TARGET_SHAPE   = (256, 256)
    MODEL_NAME     = 'efficientnet_b0'

    # Inference
    INFER_BATCH    = 8
    INFER_STRIDE   = 5
    NUM_WORKERS    = 0
    TIME_LIMIT_SEC = 7000  # ~1h 56min safety cutoff

cfg = CFG()
print('Test audio exists :', cfg.TEST_AUDIO_DIR.exists())
print('Sample sub exists :', cfg.SAMPLE_SUB.exists())
for mp in cfg.MODEL_PATHS:
    print(f'Model exists      : {Path(mp).exists()}  ({mp})')

Test audio exists : True
Sample sub exists : True
Model exists      : True  (/kaggle/input/datasets/saisamyukthan/birdclef2026-v1-efficientnet-b0-pretrained-outputs/model_fold0.pt)


In [3]:
# Rebuild class list — must match training exactly
train_df = pd.read_csv(cfg.TRAIN_CSV)
species_col = 'primary_label' if 'primary_label' in train_df.columns else 'species_code'
all_classes = sorted(train_df[species_col].unique())
NUM_CLASSES = len(all_classes)
le = LabelEncoder()
le.fit(all_classes)
class_list = le.classes_.tolist()
print(f'Classes: {NUM_CLASSES}  |  First 3: {class_list[:3]}')

Classes: 206  |  First 3: ['1161364', '116570', '1176823']


In [4]:
def load_audio_chunk(fp, sr, offset, duration):
    target = int(sr * duration)
    try:
        y, _ = librosa.load(fp, sr=sr, offset=offset, duration=duration, mono=True)
    except Exception:
        return np.zeros(target, dtype=np.float32)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y[:target].astype(np.float32)

def to_melspec(y, cfg):
    mel = librosa.feature.melspectrogram(
        y=y, sr=cfg.SAMPLE_RATE, n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH,
        n_mels=cfg.N_MELS, fmin=cfg.FMIN, fmax=cfg.FMAX, power=2.0)
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
    return mel.astype(np.float32)

def process_chunk(fp, offset, cfg):
    y = load_audio_chunk(fp, cfg.SAMPLE_RATE, offset, cfg.WINDOW_SIZE)
    spec = to_melspec(y, cfg)
    spec = cv2.resize(spec, (cfg.TARGET_SHAPE[1], cfg.TARGET_SHAPE[0]),
                      interpolation=cv2.INTER_CUBIC)
    return torch.tensor(spec).unsqueeze(0)  # (1, H, W)

class InferenceDataset(Dataset):
    def __init__(self, filepaths, cfg):
        self.cfg = cfg
        self.samples = []
        for fp in sorted(filepaths):
            stem = Path(fp).stem
            try:
                dur = sf.info(fp).duration
            except Exception:
                dur = 60.0
            for off in np.arange(0, dur, cfg.INFER_STRIDE):
                end = int(off + cfg.WINDOW_SIZE)
                self.samples.append((fp, float(off), f'{stem}_{end}'))
        print(f'{len(filepaths)} files  →  {len(self.samples)} windows')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        fp, off, row_id = self.samples[idx]
        return process_chunk(fp, off, self.cfg), row_id

print('Pipeline defined.')

Pipeline defined.


In [5]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(self.eps).pow(self.p), 1).pow(1/self.p)

class BirdCLEFModel(nn.Module):
    def __init__(self, num_classes, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.MODEL_NAME, pretrained=False,
            num_classes=0, global_pool='', in_chans=1)
        d = self.backbone.num_features
        self.pool = GeM()
        self.bn   = nn.BatchNorm1d(d)
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(d, num_classes)
    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        x = self.bn(x)
        x = self.drop(x)
        return self.fc(x)

print('Model class defined.')

Model class defined.


In [6]:
# Find test files
test_files = sorted(cfg.TEST_AUDIO_DIR.glob('*.ogg'))
if not test_files: test_files = sorted(cfg.TEST_AUDIO_DIR.glob('*.wav'))
if not test_files: test_files = sorted(cfg.TEST_AUDIO_DIR.glob('*.flac'))
print(f'Test files found: {len(test_files)}')

valid_models = [p for p in cfg.MODEL_PATHS if Path(p).exists()]
print(f'Model files found: {len(valid_models)}')

# assert len(test_files) > 0, 'No test files found. Check TEST_AUDIO_DIR.'
# assert len(valid_models) > 0, 'No model files found. Check MODEL_PATHS in CFG.'

if len(test_files) == 0:
    print('⚠️  No test files found locally — this is normal.')
    print('   Test soundscapes only appear during Kaggle official rerun.')
    print('   Proceed to Save Version → Submit to Competition.')
else:
    # Only runs during official Kaggle rerun
    dataset = InferenceDataset([str(f) for f in test_files], cfg)
    loader  = DataLoader(dataset, batch_size=cfg.INFER_BATCH,
                         shuffle=False, num_workers=cfg.NUM_WORKERS)
    all_row_ids = [s[2] for s in dataset.samples]
    ensemble    = np.zeros((len(dataset), NUM_CLASSES), dtype=np.float32)
    for i, mp in enumerate(valid_models):
        print(f'\nModel {i+1}/{len(valid_models)}: {Path(mp).name}')
        ensemble += predict_one_model(mp) / len(valid_models)
        print(f'  Elapsed: {(time.time()-START_TIME)/60:.1f} min')
    print(f'\nDone. Predictions shape: {ensemble.shape}')

    @torch.no_grad()
    def predict_one_model(model_path):
        model = BirdCLEFModel(NUM_CLASSES, cfg)
        model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True))
        model.eval()
        preds = []
        for i, (x, _) in enumerate(tqdm(loader, desc=f'  {Path(model_path).name}')):
            if i % 50 == 0 and (time.time() - START_TIME) > cfg.TIME_LIMIT_SEC:
                print('  Time limit reached — padding remaining predictions.')
                remaining = len(dataset) - len(preds) * cfg.INFER_BATCH
                preds.append(np.full((max(0, remaining), NUM_CLASSES), 0.01))
                break
            preds.append(torch.sigmoid(model(x)).numpy())
        p = np.concatenate(preds, axis=0)
        if len(p) < len(dataset):
            p = np.concatenate([p, np.full((len(dataset)-len(p), NUM_CLASSES), 0.01)])
        del model; gc.collect()
        return p
    
    for i, mp in enumerate(valid_models):
        print(f'\nModel {i+1}/{len(valid_models)}: {Path(mp).name}')
        ensemble += predict_one_model(mp) / len(valid_models)
        print(f'  Elapsed: {(time.time()-START_TIME)/60:.1f} min')
    
    print(f'\nDone. Predictions shape: {ensemble.shape}')

Test files found: 0
Model files found: 1
⚠️  No test files found locally — this is normal.
   Test soundscapes only appear during Kaggle official rerun.
   Proceed to Save Version → Submit to Competition.


In [7]:

if len(test_files) == 0:
    print('No test files — skipping submission build.')
    print('This notebook is ready to submit. Click Save Version → Submit.')
    sample_sub = pd.read_csv(cfg.SAMPLE_SUB)
    sp_cols = [c for c in sample_sub.columns if c != 'row_id']
    sample_sub[sp_cols] = 0.01
    sub_path = '/kaggle/working/submission.csv'
    sample_sub.to_csv(sub_path, index=False)
    print('No test files found - fallback submission written.')
    sys.exit(0)

else:
    sample_sub = pd.read_csv(cfg.SAMPLE_SUB)
    
    pred_df = pd.DataFrame(ensemble, columns=class_list)
    pred_df.insert(0, 'row_id', all_row_ids)
    
    submission = sample_sub[['row_id']].merge(pred_df, on='row_id', how='left')
    sp_cols = [c for c in submission.columns if c != 'row_id']
    submission[sp_cols] = submission[sp_cols].fillna(0.01)
    for col in sample_sub.columns:
        if col not in submission.columns:
            submission[col] = 0.01
    submission = submission[sample_sub.columns.tolist()]
    
    sub_path = '/kaggle/working/submission.csv'
    submission.to_csv(sub_path, index=False)
    
    # Sanity checks
    ref = sample_sub
    checks = [
        ('Row count matches',        len(submission) == len(ref)),
        ('Column count matches',     len(submission.columns) == len(ref.columns)),
        ('No NaN values',            submission.isnull().sum().sum() == 0),
        ('All values in [0,1]',      bool((submission[sp_cols].values >= 0).all() and
                                          (submission[sp_cols].values <= 1).all())),
        ('Column names match',       list(submission.columns) == list(ref.columns)),
    ]
    all_ok = True
    for desc, result in checks:
        print(f'{"✅" if result else "❌"}  {desc}')
        if not result: all_ok = False
    
    print()
    if all_ok:
        print(f'🎉 Ready to submit!  Runtime: {(time.time()-START_TIME)/60:.1f} min')
    else:
        print('⚠️  Fix failing checks before submitting.')
    
    submission.head(3)

No test files — skipping submission build.
This notebook is ready to submit. Click Save Version → Submit.
No test files found - fallback submission written.


SystemExit: 0